#YOLO26 Training Pipeline with Custom Metrics & WandB

This notebook covers:
1. **Custom detection metrics from scratch** - IoU, Precision, Recall, AP, mAP
2. **YOLO26 training** via `ultralytics` pip package
3. **WandB integration** - custom metric logging + predicted vs GT visual inspection every epoch

**Dataset expected structure (in Google Drive):**
```
split_dataset/
├── train/
│   ├── images/   (image1.jpg, image2.jpg, ...)
│   └── labels/   (image1.txt, image2.txt, ...)
├── val/
│   ├── images/
│   └── labels/
└── test/
    ├── images/
    └── labels/
```

**YOLO annotation format** (one object per line):
```
<class_idx> <cx> <cy> <w> <h>    (all values normalised 0–1)
```


## 📦 Cell 1 — Install Dependencies

In [1]:
#Install/upgrade all required packages
!pip install -q -U ultralytics          # YOLO26 lives here
!pip install -q wandb                   # Experiment tracking

#Verify ultralytics version (should be ≥ 8.4 for YOLO26)
import ultralytics
print(f'ultralytics version: {ultralytics.__version__}')
ultralytics.checks()  # GPU/ CUDA sanity check

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 42.9/112.6 GB disk)


## 📂 Cell 2 — Mount Google Drive & Configure Dataset Path

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_ROOT = '/content/drive/MyDrive/split_dataset'

import os
from pathlib import Path

TRAIN_IMAGES = os.path.join(DATASET_ROOT, 'train', 'images')
TRAIN_LABELS = os.path.join(DATASET_ROOT, 'train', 'labels')
VAL_IMAGES   = os.path.join(DATASET_ROOT, 'val',   'images')
VAL_LABELS   = os.path.join(DATASET_ROOT, 'val',   'labels')
TEST_IMAGES  = os.path.join(DATASET_ROOT, 'test',  'images')
TEST_LABELS  = os.path.join(DATASET_ROOT, 'test',  'labels')

NUM_CLASSES = 14  # indices 0–13

# Quick sanity check
for split, path in [('train/images', TRAIN_IMAGES),
                    ('val/images',   VAL_IMAGES),
                    ('test/images',  TEST_IMAGES)]:
    n = len(list(Path(path).glob('*.*')))
    print(f'{split}: {n} files')

Mounted at /content/drive
train/images: 3962 files
val/images: 600 files
test/images: 595 files


## 📥 Cell 3 — Imports

In [3]:
import os, random, shutil, warnings
import numpy as np
import cv2
import yaml
import wandb
from pathlib import Path
from copy import deepcopy
from ultralytics import YOLO

warnings.filterwarnings('ignore')

# Reproducibility seed (optional)
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print('All imports OK ✅')

All imports OK ✅


---
## 📐 Custom Detection Metrics — Implemented from Scratch

We implement the full detection metric stack:
- **IoU** — Intersection over Union between two boxes
- **Precision / Recall** — aggregated over all classes
- **AP** — Area under the Precision-Recall curve (COCO-style)
- **mAP** — Mean AP across all classes

All box coordinates are in `[x1, y1, x2, y2]` (absolute pixel) format unless stated otherwise.

## 📐 Cell 4 — IoU

In [4]:
def compute_iou(box1: np.ndarray, box2: np.ndarray) -> float:
    """
    Compute Intersection over Union (IoU) between two axis-aligned boxes.

    Parameters
    ----------
    box1, box2 : array-like, shape (4,)
        Boxes in [x1, y1, x2, y2] format (absolute pixel coordinates).

    Returns
    -------
    float
        IoU value in [0, 1].  Returns 0 for degenerate / non-overlapping boxes.
    """
    # Intersection rectangle
    ix1 = max(box1[0], box2[0])
    iy1 = max(box1[1], box2[1])
    ix2 = min(box1[2], box2[2])
    iy2 = min(box1[3], box2[3])

    intersection = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    if intersection == 0.0:
        return 0.0

    area1 = max(0.0, box1[2] - box1[0]) * max(0.0, box1[3] - box1[1])
    area2 = max(0.0, box2[2] - box2[0]) * max(0.0, box2[3] - box2[1])
    union = area1 + area2 - intersection

    if union <= 0.0:
        return 0.0

    return float(intersection / union)


def batch_iou(boxes_a: np.ndarray, boxes_b: np.ndarray) -> np.ndarray:
    """
    Vectorised IoU between every pair in boxes_a x boxes_b.

    Parameters
    ----------
    boxes_a : ndarray, shape (N, 4)
    boxes_b : ndarray, shape (M, 4)

    Returns
    -------
    ndarray, shape (N, M)  — IoU matrix
    """
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)), dtype=np.float32)

    #Broadcast to (N, M, 4)
    a = boxes_a[:, None, :]   # (N, 1, 4)
    b = boxes_b[None, :, :]   # (1, M, 4)

    ix1 = np.maximum(a[..., 0], b[..., 0])
    iy1 = np.maximum(a[..., 1], b[..., 1])
    ix2 = np.minimum(a[..., 2], b[..., 2])
    iy2 = np.minimum(a[..., 3], b[..., 3])

    inter = np.maximum(0.0, ix2 - ix1) * np.maximum(0.0, iy2 - iy1)  # (N, M)
    area_a = (boxes_a[:, 2] - boxes_a[:, 0]) * (boxes_a[:, 3] - boxes_a[:, 1])  # (N,)
    area_b = (boxes_b[:, 2] - boxes_b[:, 0]) * (boxes_b[:, 3] - boxes_b[:, 1])  # (M,)
    union = area_a[:, None] + area_b[None, :] - inter                            # (N, M)

    return np.where(union > 0, inter / union, 0.0).astype(np.float32)


print('IoU functions defined ✅')

IoU functions defined ✅


## 📐 Cell 5 — AP (Area Under the Precision-Recall Curve)

In [5]:
def compute_ap(recalls: np.ndarray, precisions: np.ndarray) -> float:
    """
    Compute Average Precision (AP) using the all-point interpolation method
    (same as PASCAL VOC 2010+ / COCO).

    Parameters
    ----------
    recalls    : 1-D array, already sorted in ascending order, values in [0, 1]
    precisions : 1-D array matching recalls

    Returns
    -------
    float — AP in [0, 1]
    """
    #Sentinel values so the curve starts at (R=0, P=0) and ends at (R=1, P=0)
    recalls    = np.concatenate(([0.0], recalls,    [1.0]))
    precisions = np.concatenate(([1.0], precisions, [0.0]))

    #Make precision monotonically non-increasing (right-to-left max)
    for i in range(len(precisions) - 2, -1, -1):
        precisions[i] = max(precisions[i], precisions[i + 1])

    #Find all recall change-points and integrate (trapezoid rule)
    change_idx = np.where(recalls[1:] != recalls[:-1])[0] + 1
    ap = float(np.sum(
        (recalls[change_idx] - recalls[change_idx - 1]) * precisions[change_idx]
    ))
    return ap


print('AP function defined ✅')

AP function defined ✅


## 📐 Cell 6 — Per-Class AP & Full mAP

In [6]:
def compute_class_ap(
    all_pred_boxes:   list,   # list[np.ndarray (N_i, 4)]  — xyxy per image
    all_pred_scores:  list,   # list[np.ndarray (N_i,)]    — confidence per image
    all_pred_labels:  list,   # list[np.ndarray (N_i,)]    — class index per image
    all_gt_boxes:     list,   # list[np.ndarray (M_i, 4)]
    all_gt_labels:    list,   # list[np.ndarray (M_i,)]
    class_id:         int,
    iou_threshold:    float = 0.5,
) -> float | None:
    """
    Compute AP for a single class across the entire dataset.

    Returns None when the class has no ground-truth instances
    (so it can be skipped in the mAP average).
    """
    #1. Collect class-specific predictions with image index bookkeeping
    flat_boxes, flat_scores, flat_img_idx = [], [], []
    gt_boxes_per_img  = []   # class-filtered GT boxes for each image
    total_gt_count    = 0

    for img_i in range(len(all_pred_boxes)):
        pred_labels = all_pred_labels[img_i]
        gt_labels   = all_gt_labels[img_i]

        #Predictions for this class in this image
        if len(pred_labels) > 0:
            mask = pred_labels == class_id
            flat_boxes.append(all_pred_boxes[img_i][mask])
            flat_scores.append(all_pred_scores[img_i][mask])
            flat_img_idx.extend([img_i] * int(mask.sum()))
        else:
            flat_boxes.append(np.empty((0, 4), dtype=np.float32))
            flat_scores.append(np.empty((0,),  dtype=np.float32))

        #GT for this class in this image
        if len(gt_labels) > 0:
            gt_mask = gt_labels == class_id
            cls_gt  = all_gt_boxes[img_i][gt_mask]
        else:
            cls_gt = np.empty((0, 4), dtype=np.float32)

        gt_boxes_per_img.append(cls_gt)
        total_gt_count += len(cls_gt)

    if total_gt_count == 0:
        return None  # class absent in this split- skip

    #2. Flatten and sort by confidence (descending)
    all_boxes  = np.concatenate(flat_boxes,  axis=0) if flat_boxes  else np.empty((0, 4))
    all_scores = np.concatenate(flat_scores, axis=0) if flat_scores else np.empty((0,))
    all_img    = np.array(flat_img_idx, dtype=np.int32)

    if len(all_boxes) == 0:
        return 0.0  #no predictions at all -> AP = 0

    sort_idx  = np.argsort(-all_scores)
    all_boxes  = all_boxes[sort_idx]
    all_scores = all_scores[sort_idx]
    all_img    = all_img[sort_idx]

    #3. Accumulate TP / FP
    tp = np.zeros(len(all_boxes), dtype=np.float32)
    fp = np.zeros(len(all_boxes), dtype=np.float32)
    matched_per_img = [set() for _ in range(len(all_pred_boxes))]

    for pred_i in range(len(all_boxes)):
        img_i   = int(all_img[pred_i])
        pred_box = all_boxes[pred_i]                       # shape (4,)
        gt_pool  = gt_boxes_per_img[img_i]                 # shape (K, 4)

        best_iou_val  = iou_threshold - 1e-9  # must exceed threshold
        best_gt_idx   = -1

        if len(gt_pool) > 0:
            ious = batch_iou(pred_box[None], gt_pool)[0]   # shape (K,)
            for gt_j, iou_val in enumerate(ious):
                if gt_j in matched_per_img[img_i]:
                    continue
                if iou_val > best_iou_val:
                    best_iou_val = iou_val
                    best_gt_idx  = gt_j

        if best_gt_idx >= 0:
            tp[pred_i] = 1.0
            matched_per_img[img_i].add(best_gt_idx)
        else:
            fp[pred_i] = 1.0

    #4. Cumulative precision & recall -> AP
    cum_tp = np.cumsum(tp)
    cum_fp = np.cumsum(fp)
    recalls    = cum_tp / (total_gt_count + 1e-9)
    precisions = cum_tp / (cum_tp + cum_fp + 1e-9)

    return compute_ap(recalls, precisions)




def compute_map(
    all_pred_boxes:  list,
    all_pred_scores: list,
    all_pred_labels: list,
    all_gt_boxes:    list,
    all_gt_labels:   list,
    num_classes:     int   = 14,
    iou_threshold:   float = 0.5,
) -> tuple:
    """
    Compute mAP and per-class AP.

    Parameters
    ----------
    all_pred_boxes   : list of (N_i, 4) arrays  — xyxy
    all_pred_scores  : list of (N_i,)  arrays  — confidence
    all_pred_labels  : list of (N_i,)  arrays  — class index
    all_gt_boxes     : list of (M_i, 4) arrays  — xyxy
    all_gt_labels    : list of (M_i,)  arrays  — class index
    num_classes      : total number of classes
    iou_threshold    : IoU threshold for TP/FP assignment

    Returns
    -------
    (mAP, per_class_ap_dict)
        mAP             — float, mean over classes that have ≥1 GT instance
        per_class_ap    — dict {class_id: ap_float}
    """
    per_class_ap = {}

    for cls_id in range(num_classes):
        ap = compute_class_ap(
            all_pred_boxes, all_pred_scores, all_pred_labels,
            all_gt_boxes,   all_gt_labels,
            class_id=cls_id, iou_threshold=iou_threshold,
        )
        if ap is not None:          # None = class not present in this split
            per_class_ap[cls_id] = ap

    mAP = float(np.mean(list(per_class_ap.values()))) if per_class_ap else 0.0
    return mAP, per_class_ap


# ─────────────────────────────────────────────────────────────────────────────

def compute_precision_recall(
    all_pred_boxes:  list,
    all_pred_scores: list,
    all_pred_labels: list,
    all_gt_boxes:    list,
    all_gt_labels:   list,
    iou_threshold:   float = 0.5,
    conf_threshold:  float = 0.25,
) -> tuple:
    """
    Compute micro-averaged Precision and Recall across all classes.

    Returns
    -------
    (precision, recall)  — floats in [0, 1]
    """
    total_tp = total_fp = total_fn = 0

    for img_i in range(len(all_pred_boxes)):
        pred_boxes  = all_pred_boxes[img_i]
        pred_scores = all_pred_scores[img_i]
        pred_labels = all_pred_labels[img_i]
        gt_boxes    = all_gt_boxes[img_i]
        gt_labels   = all_gt_labels[img_i]

        # Apply confidence threshold
        if len(pred_scores) > 0:
            conf_mask   = pred_scores >= conf_threshold
            pred_boxes  = pred_boxes[conf_mask]
            pred_labels = pred_labels[conf_mask]

        matched_gt = set()

        for pred_i in range(len(pred_boxes)):
            pred_cls = pred_labels[pred_i]

            # Only match GT of same class
            cls_gt_mask = gt_labels == pred_cls
            cls_gt_boxes = gt_boxes[cls_gt_mask]
            cls_gt_orig_idx = np.where(cls_gt_mask)[0]

            best_iou_val = iou_threshold - 1e-9
            best_gt_orig = -1

            if len(cls_gt_boxes) > 0:
                ious = batch_iou(pred_boxes[pred_i:pred_i+1], cls_gt_boxes)[0]
                for local_j, iou_val in enumerate(ious):
                    orig_j = cls_gt_orig_idx[local_j]
                    if orig_j in matched_gt:
                        continue
                    if iou_val > best_iou_val:
                        best_iou_val = iou_val
                        best_gt_orig = orig_j

            if best_gt_orig >= 0:
                total_tp += 1
                matched_gt.add(best_gt_orig)
            else:
                total_fp += 1

        total_fn += len(gt_boxes) - len(matched_gt)

    precision = total_tp / (total_tp + total_fp + 1e-9)
    recall    = total_tp / (total_tp + total_fn + 1e-9)
    return float(precision), float(recall)


print('mAP / Precision / Recall functions defined ✅')

mAP / Precision / Recall functions defined ✅


## 📐 Cell 7 — Metric Sanity Check with Synthetic Data

In [7]:
#Sanity check: perfect predictions should yield mAP = 1.0
print('=== Sanity Check: Perfect Predictions ===')
gt_b = [np.array([[10, 10, 50, 50], [60, 60, 100, 100]], dtype=np.float32)]
gt_l = [np.array([0, 1])]
pr_b = [np.array([[10, 10, 50, 50], [60, 60, 100, 100]], dtype=np.float32)]
pr_s = [np.array([0.99, 0.98])]
pr_l = [np.array([0, 1])]

map_val, per_cls = compute_map(pr_b, pr_s, pr_l, gt_b, gt_l, num_classes=2)
p, r = compute_precision_recall(pr_b, pr_s, pr_l, gt_b, gt_l)
print(f'  mAP  = {map_val:.4f}  (expected ≈ 1.0)')
print(f'  P    = {p:.4f}  (expected ≈ 1.0)')
print(f'  R    = {r:.4f}  (expected ≈ 1.0)')
assert map_val > 0.99, 'mAP should be ~1 for perfect preds'

print()
print('=== Sanity Check: No Predictions ===')
pr_b2 = [np.empty((0, 4), dtype=np.float32)]
pr_s2 = [np.empty((0,),   dtype=np.float32)]
pr_l2 = [np.empty((0,),   dtype=np.int32)]
map_val2, _ = compute_map(pr_b2, pr_s2, pr_l2, gt_b, gt_l, num_classes=2)
p2, r2 = compute_precision_recall(pr_b2, pr_s2, pr_l2, gt_b, gt_l)
print(f'  mAP  = {map_val2:.4f}  (expected 0.0)')
print(f'  P    = {p2:.4f}  (expected 0.0)')
print(f'  R    = {r2:.4f}  (expected 0.0)')
assert map_val2 == 0.0, 'mAP should be 0 for no predictions'

print()
print('=== Sanity Check: Single IoU Call ===')
iou_same = compute_iou([0, 0, 10, 10], [0, 0, 10, 10])
iou_none = compute_iou([0, 0, 5, 5], [10, 10, 20, 20])
iou_half = compute_iou([0, 0, 10, 10], [5, 0, 15, 10])
print(f'  IoU same box  = {iou_same:.4f}  (expected 1.0)')
print(f'  IoU no overlap= {iou_none:.4f}  (expected 0.0)')
print(f'  IoU half      = {iou_half:.4f}  (expected 0.333)')
assert abs(iou_same - 1.0) < 1e-5
assert abs(iou_none - 0.0) < 1e-5
assert abs(iou_half - 1/3) < 1e-3

print()
print('All sanity checks passed ✅')

=== Sanity Check: Perfect Predictions ===
  mAP  = 1.0000  (expected ≈ 1.0)
  P    = 1.0000  (expected ≈ 1.0)
  R    = 1.0000  (expected ≈ 1.0)

=== Sanity Check: No Predictions ===
  mAP  = 0.0000  (expected 0.0)
  P    = 0.0000  (expected 0.0)
  R    = 0.0000  (expected 0.0)

=== Sanity Check: Single IoU Call ===
  IoU same box  = 1.0000  (expected 1.0)
  IoU no overlap= 0.0000  (expected 0.0)
  IoU half      = 0.3333  (expected 0.333)

All sanity checks passed ✅


---
## 🗂️ Dataset YAML Configuration

## 🗂️ Cell 8 — Create `dataset.yaml` for YOLO26

In [8]:

CLASS_NAMES = [
    'class_0', 'class_1', 'class_2', 'class_3',
    'class_4', 'class_5', 'class_6', 'class_7',
    'class_8', 'class_9', 'class_10', 'class_11',
    'class_12', 'class_13',
]

YAML_PATH = '/content/dataset.yaml'

dataset_cfg = {
    'path': DATASET_ROOT,
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc':    NUM_CLASSES,
    'names': CLASS_NAMES,
}

with open(YAML_PATH, 'w') as f:
    yaml.dump(dataset_cfg, f, default_flow_style=False)

print(f'Dataset YAML written to {YAML_PATH}')
print()
with open(YAML_PATH) as f:
    print(f.read())

Dataset YAML written to /content/dataset.yaml

names:
- class_0
- class_1
- class_2
- class_3
- class_4
- class_5
- class_6
- class_7
- class_8
- class_9
- class_10
- class_11
- class_12
- class_13
nc: 14
path: /content/drive/MyDrive/split_dataset
test: test/images
train: train/images
val: val/images



---
## 🪄 WandB Login & Helper Utilities

## 🪄 Cell 9 — WandB Login

In [9]:
# You will be prompted for your WandB API key.
# Alternatively set:  os.environ['WANDB_API_KEY'] = 'your_key_here'
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dragostrandafir443 (dragostrandafir443-babes) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 🪄 Cell 10 — Helper: Parse YOLO Labels and Run Inference

In [10]:
def parse_yolo_label(label_path: Path, img_w: int, img_h: int):
    """
    Parse a YOLO-format .txt annotation file.

    Returns
    -------
    boxes  : ndarray (M, 4) in xyxy absolute pixels
    labels : ndarray (M,)   class indices
    """
    boxes, labels = [], []
    if not label_path.exists():
        return np.empty((0, 4), np.float32), np.array([], np.int32)

    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls, cx, cy, bw, bh = (float(x) for x in parts[:5])
            x1 = (cx - bw / 2) * img_w
            y1 = (cy - bh / 2) * img_h
            x2 = (cx + bw / 2) * img_w
            y2 = (cy + bh / 2) * img_h
            boxes.append([x1, y1, x2, y2])
            labels.append(int(cls))

    boxes  = np.array(boxes,  dtype=np.float32) if boxes  else np.empty((0, 4), np.float32)
    labels = np.array(labels, dtype=np.int32)   if labels else np.array([], np.int32)
    return boxes, labels


def collect_val_predictions(
    model_for_inference,          # any callable returning Ultralytics Results
    val_images_dir:  str,
    val_labels_dir:  str,
    max_images:      int   = 150,
    conf_threshold:  float = 0.001,  # low threshold → keep for mAP sweep
    img_extensions=  ('*.jpg', '*.jpeg', '*.png', '*.bmp'),
):
    """
    Run inference on up to `max_images` validation images and collect
    predictions + ground-truth in the list-of-arrays format used by
    compute_map / compute_precision_recall.
    """
    img_paths = []
    for ext in img_extensions:
        img_paths.extend(Path(val_images_dir).glob(ext))
    img_paths = sorted(img_paths)[:max_images]

    all_pred_boxes, all_pred_scores, all_pred_labels = [], [], []
    all_gt_boxes,   all_gt_labels                    = [], []

    for img_path in img_paths:
        # Load image to get spatial dimensions
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img_h, img_w = img.shape[:2]

        #Ground truth
        label_path = Path(val_labels_dir) / (img_path.stem + '.txt')
        gt_boxes, gt_labels = parse_yolo_label(label_path, img_w, img_h)
        all_gt_boxes.append(gt_boxes)
        all_gt_labels.append(gt_labels)

        # Predictions
        try:
            results = model_for_inference(str(img_path), verbose=False, conf=conf_threshold)
            res = results[0]
            if res.boxes is not None and len(res.boxes) > 0:
                pred_boxes  = res.boxes.xyxy.cpu().numpy().astype(np.float32)
                pred_scores = res.boxes.conf.cpu().numpy().astype(np.float32)
                pred_labels = res.boxes.cls.cpu().numpy().astype(np.int32)
            else:
                pred_boxes  = np.empty((0, 4), np.float32)
                pred_scores = np.empty((0,),   np.float32)
                pred_labels = np.array([],     dtype=np.int32)
        except Exception as e:
            print(f'  [warn] inference failed on {img_path.name}: {e}')
            pred_boxes  = np.empty((0, 4), np.float32)
            pred_scores = np.empty((0,),   np.float32)
            pred_labels = np.array([],     dtype=np.int32)

        all_pred_boxes.append(pred_boxes)
        all_pred_scores.append(pred_scores)
        all_pred_labels.append(pred_labels)

    return (
        all_pred_boxes, all_pred_scores, all_pred_labels,
        all_gt_boxes,   all_gt_labels,
        img_paths,
    )


print('Helper functions defined ✅')

Helper functions defined ✅


## 🪄 Cell 11 — Helper: Draw Boxes on Image (for Visual Inspection)

In [11]:
# Distinct colour palette-one per class (BGR for OpenCV)
_PALETTE = [
    (0, 255, 0),   (255, 0, 0),   (0, 0, 255),   (255, 255, 0),
    (0, 255, 255), (255, 0, 255), (128, 255, 0),  (0, 128, 255),
    (255, 128, 0), (128, 0, 255), (0, 255, 128),  (255, 0, 128),
    (200, 200, 0), (0, 200, 200),
]

def class_color(cls_id: int):
    return _PALETTE[cls_id % len(_PALETTE)]


def draw_boxes(
    image:   np.ndarray,   # RGB, shape (H, W, 3)
    boxes:   np.ndarray,   # (N, 4) xyxy absolute
    labels:  np.ndarray,   # (N,)
    scores:  np.ndarray | None = None,
    class_names: list = None,
    thickness: int = 2,
) -> np.ndarray:
    """
    Draw bounding boxes with class labels (and optionally scores) on an RGB image.
    Returns a new copy — original is not modified.
    """
    out = image.copy()
    for i, (box, cls) in enumerate(zip(boxes, labels)):
        x1, y1, x2, y2 = map(int, box)
        color = class_color(cls)[::-1]  # BGR→RGB

        cv2.rectangle(out, (x1, y1), (x2, y2), color, thickness)

        label_str = class_names[cls] if class_names else f'cls{cls}'
        if scores is not None and i < len(scores):
            label_str += f' {scores[i]:.2f}'

        #Small filled rectangle behind text for legibility
        (tw, th), _ = cv2.getTextSize(label_str, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        ty = max(y1 - 4, th + 4)
        cv2.rectangle(out, (x1, ty - th - 4), (x1 + tw + 2, ty), color, -1)
        cv2.putText(out, label_str, (x1 + 1, ty - 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
    return out


def make_gt_pred_panel(
    img_path:        Path,
    model_for_infer,
    val_labels_dir:  str,
    class_names:     list,
    conf_threshold:  float = 0.25,
    target_width:    int   = 640,
) -> np.ndarray | None:
    """
    Build a side-by-side panel [GT | PRED] for a single image.
    Returns an RGB numpy array or None on failure.
    """
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        return None
    img_h, img_w = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    #Ground truth side
    label_path = Path(val_labels_dir) / (img_path.stem + '.txt')
    gt_boxes, gt_labels = parse_yolo_label(label_path, img_w, img_h)
    gt_panel = draw_boxes(img_rgb, gt_boxes, gt_labels,
                          class_names=class_names)

    # Prediction side
    try:
        results = model_for_infer(str(img_path), verbose=False, conf=conf_threshold)
        res = results[0]
        if res.boxes is not None and len(res.boxes) > 0:
            pred_boxes  = res.boxes.xyxy.cpu().numpy().astype(np.float32)
            pred_scores = res.boxes.conf.cpu().numpy().astype(np.float32)
            pred_labels = res.boxes.cls.cpu().numpy().astype(np.int32)
        else:
            pred_boxes = pred_scores = pred_labels = np.array([])
    except Exception:
        pred_boxes = pred_scores = pred_labels = np.array([])

    pred_panel = draw_boxes(
        img_rgb,
        pred_boxes if len(pred_boxes) else np.empty((0, 4)),
        pred_labels.astype(np.int32) if len(pred_labels) else np.array([], dtype=np.int32),
        scores=pred_scores if len(pred_scores) else None,
        class_names=class_names,
    )

    #Add text banner
    banner_h = 28
    for panel, label in [(gt_panel, 'GROUND TRUTH'), (pred_panel, 'PREDICTION')]:
        banner = np.zeros((banner_h, panel.shape[1], 3), np.uint8)
        cv2.putText(banner, label, (6, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
        panel[:] = np.concatenate([banner, panel[:-banner_h]], axis=0)

    combined = np.concatenate([gt_panel, pred_panel], axis=1)  # side by side

    # Resize to target width for uniform WandB thumbnails
    scale = target_width / combined.shape[1]
    if scale != 1.0:
        new_h = int(combined.shape[0] * scale)
        combined = cv2.resize(combined, (target_width, new_h), interpolation=cv2.INTER_AREA)

    return combined


print('Drawing helpers defined ✅')

Drawing helpers defined ✅


---
## 🔁 WandB Callback — Custom Metrics + Visual Inspection Every Epoch

## 🔁 Cell 12 — Build the WandB Callback

In [12]:
def build_wandb_callbacks(
    val_images_dir:    str,
    val_labels_dir:    str,
    class_names:       list,
    num_classes:       int   = 14,
    iou_threshold:     float = 0.5,
    max_metric_imgs:   int   = 150,   #images used for metric computation
    num_vis_images:    int   = 8,     # images shown in GT-vs-Pred panel
    vis_conf:          float = 0.25,  #confidence for visual panels
    metric_log_freq:   int   = 1,     # log every N epochs (1 = every epoch)
):
    """
    Returns two callbacks to register with YOLO26:
        on_train_start  — initialises the WandB run
        on_fit_epoch_end — computes & logs custom metrics + visual inspection
    """

    # Pre-select a fixed set of validation images for visual inspection
    all_val_imgs = sorted(
        p for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp')
        for p in Path(val_images_dir).glob(ext)
    )
    # Deterministically pick diverse images for the panel
    step = max(1, len(all_val_imgs) // num_vis_images)
    vis_img_paths = all_val_imgs[::step][:num_vis_images]

    def on_train_start(trainer):
        """Initialise WandB run with training hyperparameters."""
        if wandb.run is None:
            wandb.init(
                project = 'yolo26-custom-dataset',
                name    = 'yolo26n-training',
                config  = dict(trainer.args),
                tags    = ['yolo26', 'object-detection'],
            )
        print('[WandB] Run initialised:', wandb.run.name)

    def on_fit_epoch_end(trainer):
        """
        Called after each complete epoch (train + val).

        Logs:
          • Ultralytics native metrics (loss, mAP50, mAP50-95, P, R)
          • Our custom metrics (mAP@0.5, per-class AP, Precision, Recall)
          • GT-vs-Prediction visual panels for a fixed image subset
        """
        epoch = trainer.epoch  # 0-indexed

        #1. Native Ultralytics metrics
        native_metrics = {}
        if hasattr(trainer, 'metrics') and trainer.metrics:
            for k, v in trainer.metrics.items():
                try:
                    native_metrics[f'ultralytics/{k}'] = float(v)
                except (TypeError, ValueError):
                    pass

        # Also log learning rate
        if hasattr(trainer, 'optimizer') and trainer.optimizer:
            native_metrics['train/lr'] = trainer.optimizer.param_groups[0]['lr']

        #2. Get the inference model (EMA if available
        # We wrap it in a YOLO object to reuse Ultralytics predict pipeline
        try:
            import torch
            if hasattr(trainer, 'ema') and trainer.ema is not None:
                infer_model_nn = trainer.ema.ema
            else:
                infer_model_nn = trainer.model

            # Put into eval mode temporarily
            was_training = infer_model_nn.training
            infer_model_nn.eval()

            # Wrap in YOLO for the predict() pipeline
            infer_yolo = YOLO(trainer.best if Path(str(trainer.best)).exists()
                              else trainer.last, verbose=False)

        except Exception as e:
            print(f'[WandB callback] Could not load inference model: {e}')
            wandb.log({**native_metrics, 'epoch': epoch})
            return

        #3. Custom metrics (skip every metric_log_freq epochs)
        custom_metrics = {}
        if (epoch % metric_log_freq) == 0:
            try:
                (
                    all_pred_boxes, all_pred_scores, all_pred_labels,
                    all_gt_boxes, all_gt_labels, _,
                ) = collect_val_predictions(
                    infer_yolo, val_images_dir, val_labels_dir,
                    max_images=max_metric_imgs, conf_threshold=0.001,
                )

                mAP_val, per_cls_ap = compute_map(
                    all_pred_boxes, all_pred_scores, all_pred_labels,
                    all_gt_boxes,   all_gt_labels,
                    num_classes=num_classes, iou_threshold=iou_threshold,
                )
                precision, recall = compute_precision_recall(
                    all_pred_boxes, all_pred_scores, all_pred_labels,
                    all_gt_boxes,   all_gt_labels,
                    iou_threshold=iou_threshold, conf_threshold=0.25,
                )
                f1 = 2 * precision * recall / (precision + recall + 1e-9)

                custom_metrics = {
                    'custom/mAP@0.5':  mAP_val,
                    'custom/precision': precision,
                    'custom/recall':    recall,
                    'custom/F1':        f1,
                }
                for cls_id, ap in per_cls_ap.items():
                    cls_name = class_names[cls_id] if cls_id < len(class_names) else f'cls{cls_id}'
                    custom_metrics[f'custom/AP/{cls_name}'] = ap

                print(f'[Epoch {epoch}] Custom mAP@0.5={mAP_val:.4f}  '
                      f'P={precision:.4f}  R={recall:.4f}  F1={f1:.4f}')

            except Exception as e:
                print(f'[WandB callback] Custom metrics failed at epoch {epoch}: {e}')

        #4. Visual inspection - GT vs Prediction panels
        wandb_images = []
        for img_path in vis_img_paths:
            try:
                panel = make_gt_pred_panel(
                    img_path, infer_yolo, val_labels_dir,
                    class_names=class_names,
                    conf_threshold=vis_conf,
                )
                if panel is not None:
                    wandb_images.append(wandb.Image(
                        panel,
                        caption=f'Ep{epoch} | {img_path.name}  '
                                f'[left=GT  right=PRED  conf≥{vis_conf}]',
                    ))
            except Exception as e:
                print(f'[WandB callback] Panel failed for {img_path.name}: {e}')

        if wandb_images:
            custom_metrics['visual/gt_vs_pred'] = wandb_images

        #5. Restore training mode & log everything
        try:
            if was_training:
                infer_model_nn.train()
        except Exception:
            pass

        wandb.log({**native_metrics, **custom_metrics, 'epoch': epoch})

    def on_train_end(trainer):
        """Finish WandB run cleanly."""
        print('[WandB] Training complete — finishing run.')
        wandb.finish()

    return on_train_start, on_fit_epoch_end, on_train_end


print('WandB callback factory defined ✅')

WandB callback factory defined ✅


---
## 🏋️ Train YOLO26

> The smallest variant `yolo26n` (Nano) is used by default — fastest to train.
> Swap to `yolo26s`, `yolo26m`, `yolo26l`, or `yolo26x` for larger models.

## 🏋️ Cell 13 — Configure Training Parameters

In [13]:

TRAIN_CONFIG = dict(
    data        = YAML_PATH,
    epochs      = 50,          # number of training epochs
    imgsz       = 640,         # input resolution
    batch       = 16,          # batch size (reduce to 8 if OOM)
    device      = 0,           # 0 = first GPU; 'cpu' for CPU only
    workers     = 2,
    patience    = 20,          # early stopping patience
    project     = '/content/runs',
    name        = 'yolo26n_custom',
    exist_ok    = True,
    # ── Augmentation ───────────────────────────────────────────────────
    # mosaic      = 1.0,
    # mixup       = 0.1,
    # degrees     = 5.0,
    # translate   = 0.1,
    # scale       = 0.5,
    # flipud      = 0.0,
    # fliplr      = 0.5,
    # ── Loss weights ───────────────────────────────────────────────────
    box         = 7.5,
    cls         = 0.5,
    dfl         = 1.5,
    # ── WandB ──────────────────────────────────────────────────────────
    # Disable the built-in wandb integration so our custom callback
    # is the sole source of WandB logging (avoids duplicate runs)
)
# ─────────────────────────────────────────────────────────────────────────────

print('Training config:')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k}: {v}')

Training config:
  data: /content/dataset.yaml
  epochs: 50
  imgsz: 640
  batch: 16
  device: 0
  workers: 2
  patience: 20
  project: /content/runs
  name: yolo26n_custom
  exist_ok: True
  box: 7.5
  cls: 0.5
  dfl: 1.5


## 🏋️ Cell 14 — Initialise Model & Register Callbacks

In [14]:
# Disable the built-in WandB integration (we use our own)
os.environ['WANDB_MODE'] = 'disabled'     # turns off auto-init inside ultralytics

# Load YOLO26 Nano pretrained on COCO — weights auto-download on first run
model = YOLO('yolo26n.pt')

# Build our three callbacks
cb_train_start, cb_epoch_end, cb_train_end = build_wandb_callbacks(
    val_images_dir  = VAL_IMAGES,
    val_labels_dir  = VAL_LABELS,
    class_names     = CLASS_NAMES,
    num_classes     = NUM_CLASSES,
    iou_threshold   = 0.5,
    max_metric_imgs = 150,   # ↑ for more accurate mAP; ↓ to save time
    num_vis_images  = 8,     # GT vs Pred panels logged to WandB
    vis_conf        = 0.25,
    metric_log_freq = 1,     # compute custom metrics every epoch
)

# Re-enable wandb for OUR run
os.environ.pop('WANDB_MODE', None)

# Register callbacks
model.add_callback('on_train_start',    cb_train_start)
model.add_callback('on_fit_epoch_end',  cb_epoch_end)
model.add_callback('on_train_end',      cb_train_end)

print('YOLO26n loaded and callbacks registered ✅')

YOLO26n loaded and callbacks registered ✅


## 🏋️ Cell 15 — Start Training

In [15]:
# ── Train! ────────────────────────────────────────────────────────────────────
# The WandB run is initialised inside on_train_start.
# After each epoch:
#   • custom mAP, Precision, Recall, F1, per-class AP  →  WandB scalars
#   • GT vs Predicted bounding box panels              →  WandB images
# After training:
#   • WandB run is finished cleanly

results = model.train(**TRAIN_CONFIG)

print('Training finished!')
print('Best checkpoint:', model.trainer.best)
print('Last checkpoint:', model.trainer.last)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n_custom, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=20, per

[WandB] Run initialised: yolo26n-training
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/runs/yolo26n_custom
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50      2.76G      1.267      5.523    0.01749         35        640: 100% ━━━━━━━━━━━━ 248/248 2.1it/s 2:01
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 1.3it/s 14.3s
                   all        600       1314      0.851      0.038     0.0337     0.0211
[Epoch 0] Custom mAP@0.5=0.0361  P=0.2539  R=0.1283  F1=0.1704

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50      3.22G      1.178      4.227    0.01634         40        640: 100% ━━━━━━━━━━━━ 248/248 2.5it/s 1:37
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.6it/s 7.2s
                

custom/AP/class_10,▁▂▅▄▅▇▇▇▇▇▇▇▇▇▇██▇▇▇█████▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
custom/AP/class_11,▁▂▃▄▄▄▆▅▅▇▆▇▇▇▇▆▆▇▇▇▇▆▆▆▆▇▇█████████████
custom/AP/class_12,▃▁▅▅▆▆▅█▇▇▇▇▆▆▆▅▅▅██▄▆▆▆▆▆▆█████████████
custom/AP/class_13,▁▄▄▇▆▆▆▆▇▇▇▇██████▆▇▇▇▇▇▇██▆▆▆▆▆▆▆▆▆▆▆▆▆
custom/AP/class_2,▁▃▃▃▄▄▄▄▅▅▄▇▆▆▅▅███▄▄▆▅▅▅▅█▆▆▆▆▆▆▆▆▆▆▆▆▆
custom/AP/class_3,▁▁▁▁▃▁▁▂▄▄▅▄▇▇▄▄███▃▅████▇▇▆▆▆▆▆▆▆▆▆▆▆▆▆
custom/AP/class_5,▁▁▁▁▁▁▁▁▁▁▁▂▂▂██▂▂▂▂▂█▃▃▃▂▃▃▃▃▃▃▃▃▃▃▃▃▃▃
custom/AP/class_6,▁▁▄▅▄▆▆▆▇▇▆▅▆▄▄▅▅▇▅▅▇██████▆▆▆▆▆▆▆▆▆▆▆▆▆
custom/AP/class_7,▁▁▁▂▂██▁▁▁▂▃▄▄▄▄▂▂▂▂█▅▇▇▇▇▇▇▄▄▄▄▄▄▄▄▄▄▄▄
custom/AP/class_8,▁▄▁▁▁█▁▇▇▃▄▃▃▃▃▁▁▁▅▃▃▁▂▂▂▂▁▃▃▃▃▃▃▃▃▃▃▃▃▃
+14,...


Training finished!
Best checkpoint: /content/runs/yolo26n_custom/weights/best.pt
Last checkpoint: /content/runs/yolo26n_custom/weights/last.pt


---
## 🧪 Post-Training Evaluation

Load the best checkpoint and run a final evaluation with our custom metrics on **val** and **test** sets.

## 🧪 Cell 16 — Load Best Model

In [16]:
best_ckpt = str(model.trainer.best)
print(f'Loading best checkpoint: {best_ckpt}')

best_model = YOLO(best_ckpt)
print('Model loaded ✅')

Loading best checkpoint: /content/runs/yolo26n_custom/weights/best.pt
Model loaded ✅


## 🧪 Cell 17 — Ultralytics Native Validation

In [17]:
# Ultralytics' built-in validator
native_val = best_model.val(data=YAML_PATH, split='val', verbose=True)
print()
print('── Ultralytics Native Val Metrics ──')
print(f'  mAP@0.5      : {native_val.box.map50:.4f}')
print(f'  mAP@0.5:0.95 : {native_val.box.map:.4f}')
print(f'  Precision    : {native_val.box.mp:.4f}')
print(f'  Recall       : {native_val.box.mr:.4f}')

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,377,566 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.3 ms, read: 21.3±11.6 MB/s, size: 38.5 KB)
val: Scanning /content/drive/MyDrive/split_dataset/val/labels.cache... 600 images, 117 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 600/600 167.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 2.6it/s 14.7s
                   all        600       1314      0.544      0.484       0.52      0.306
               class_0         57         58       0.33      0.724      0.424      0.273
               class_1         18         18      0.554      0.611      0.699      0.452
               class_2         19         19      0.421      0.307      0.294      0.202
               class_3         18         24      0.457      0.208      0.264     0.0743
               class_4     

## 🧪 Cell 18 — Custom Metrics on Validation Set

In [18]:
print('Running custom metrics on validation set...')
(
    val_pred_boxes, val_pred_scores, val_pred_labels,
    val_gt_boxes,   val_gt_labels,   _,
) = collect_val_predictions(
    best_model, VAL_IMAGES, VAL_LABELS,
    max_images=9999,        # use all val images for final eval
    conf_threshold=0.001,
)

val_mAP, val_per_cls = compute_map(
    val_pred_boxes, val_pred_scores, val_pred_labels,
    val_gt_boxes,   val_gt_labels,
    num_classes=NUM_CLASSES, iou_threshold=0.5,
)
val_P, val_R = compute_precision_recall(
    val_pred_boxes, val_pred_scores, val_pred_labels,
    val_gt_boxes,   val_gt_labels,
    iou_threshold=0.5, conf_threshold=0.25,
)
val_F1 = 2 * val_P * val_R / (val_P + val_R + 1e-9)

print()
print('── Custom Val Metrics (mAP@IoU=0.5) ──')
print(f'  mAP@0.5   : {val_mAP:.4f}')
print(f'  Precision : {val_P:.4f}')
print(f'  Recall    : {val_R:.4f}')
print(f'  F1        : {val_F1:.4f}')
print()
print('── Per-Class AP ──')
for cls_id, ap in sorted(val_per_cls.items()):
    name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else f'cls{cls_id}'
    bar  = '█' * int(ap * 30)
    print(f'  {name:>12s} [{bar:<30s}] {ap:.4f}')

Running custom metrics on validation set...

── Custom Val Metrics (mAP@IoU=0.5) ──
  mAP@0.5   : 0.4411
  Precision : 0.4783
  Recall    : 0.5289
  F1        : 0.5023

── Per-Class AP ──
       class_0 [██████████                    ] 0.3634
       class_1 [███████████████████████       ] 0.7791
       class_2 [█████                         ] 0.1900
       class_3 [███████                       ] 0.2391
       class_4 [█████████████████             ] 0.5947
       class_5 [███                           ] 0.1000
       class_6 [████████████                  ] 0.4109
       class_7 [██████████                    ] 0.3663
       class_8 [█████████                     ] 0.3173
       class_9 [██████                        ] 0.2052
      class_10 [████████████████████████      ] 0.8106
      class_11 [████████████████████          ] 0.6837
      class_12 [█████████████████             ] 0.5823
      class_13 [████████████████              ] 0.5335


## 🧪 Cell 19 — Custom Metrics on Test Set

In [19]:
print('Running custom metrics on test set...')
(
    tst_pred_boxes, tst_pred_scores, tst_pred_labels,
    tst_gt_boxes,   tst_gt_labels,   _,
) = collect_val_predictions(
    best_model, TEST_IMAGES, TEST_LABELS,
    max_images=9999, conf_threshold=0.001,
)

tst_mAP, tst_per_cls = compute_map(
    tst_pred_boxes, tst_pred_scores, tst_pred_labels,
    tst_gt_boxes,   tst_gt_labels,
    num_classes=NUM_CLASSES, iou_threshold=0.5,
)
tst_P, tst_R = compute_precision_recall(
    tst_pred_boxes, tst_pred_scores, tst_pred_labels,
    tst_gt_boxes,   tst_gt_labels,
    iou_threshold=0.5, conf_threshold=0.25,
)
tst_F1 = 2 * tst_P * tst_R / (tst_P + tst_R + 1e-9)

print()
print('── Custom Test Metrics (mAP@IoU=0.5) ──')
print(f'  mAP@0.5   : {tst_mAP:.4f}')
print(f'  Precision : {tst_P:.4f}')
print(f'  Recall    : {tst_R:.4f}')
print(f'  F1        : {tst_F1:.4f}')
print()
print('── Per-Class AP ──')
for cls_id, ap in sorted(tst_per_cls.items()):
    name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else f'cls{cls_id}'
    bar  = '█' * int(ap * 30)
    print(f'  {name:>12s} [{bar:<30s}] {ap:.4f}')

Running custom metrics on test set...

── Custom Test Metrics (mAP@IoU=0.5) ──
  mAP@0.5   : 0.5117
  Precision : 0.4948
  Recall    : 0.5644
  F1        : 0.5273

── Per-Class AP ──
       class_0 [███████████                   ] 0.3785
       class_1 [█████████████████████         ] 0.7247
       class_2 [███████                       ] 0.2422
       class_3 [███████████                   ] 0.3701
       class_4 [█████████████████             ] 0.5780
       class_5 [██████████████████████████████] 1.0000
       class_6 [█████████                     ] 0.3282
       class_7 [██████████                    ] 0.3640
       class_8 [█████████████                 ] 0.4400
       class_9 [███████████                   ] 0.3951
      class_10 [█████████████████             ] 0.5821
      class_11 [███████████████████           ] 0.6639
      class_12 [████████████████              ] 0.5429
      class_13 [████████████████              ] 0.5537


## 🧪 Cell 20 — Log Final Metrics to WandB

In [20]:
# Open a fresh WandB run to store the final evaluation results
with wandb.init(
    project='yolo26-custom-dataset',
    name='final-evaluation',
    tags=['yolo26', 'evaluation'],
) as run:

    log_payload = {
        # Val
        'final/val_mAP@0.5':    val_mAP,
        'final/val_precision':   val_P,
        'final/val_recall':      val_R,
        'final/val_F1':          val_F1,
        # Test
        'final/test_mAP@0.5':   tst_mAP,
        'final/test_precision':  tst_P,
        'final/test_recall':     tst_R,
        'final/test_F1':         tst_F1,
    }

    # Per-class AP table
    ap_table = wandb.Table(columns=['class', 'val_AP', 'test_AP'])
    for cls_id in range(NUM_CLASSES):
        name = CLASS_NAMES[cls_id]
        v_ap = val_per_cls.get(cls_id, None)
        t_ap = tst_per_cls.get(cls_id, None)
        if v_ap is not None or t_ap is not None:
            ap_table.add_data(name, v_ap, t_ap)

    log_payload['final/per_class_AP_table'] = ap_table
    wandb.log(log_payload)

print('Final metrics logged to WandB ✅')
print(f'View your run at: {run.url}')

final/test_F1,▁
final/test_mAP@0.5,▁
final/test_precision,▁
final/test_recall,▁
final/val_F1,▁
final/val_mAP@0.5,▁
final/val_precision,▁
final/val_recall,▁
final/test_F1,0.52732
final/test_mAP@0.5,0.51166
final/test_precision,0.49479


Final metrics logged to WandB ✅
View your run at: https://wandb.ai/dragostrandafir443-babes/yolo26-custom-dataset/runs/tno3hmgf
